# MayzCats V1 — YouTube Shorts Automation

Open this notebook in Colab, fill Drive `secrets/.env`, and add the YouTube Desktop OAuth `client_secret.json` as described in README. The notebook clones MayzCats from GitHub; Drive keeps only persistent runtime data. Set `RUN_MODE`, then use **Runtime → Run all**. Upload privacy comes from `config/pipeline.yaml`.

In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

from google.colab import drive

# new: start a new video; resume_latest: continue the newest failed run; resume: use RESUME_RUN_ID
RUN_MODE = 'new'
RESUME_RUN_ID = ''
# True rebuilds only the video/subtitles while reusing cached media, music, and paid TTS.
RERENDER_ON_RESUME = False

print('[MayzCats] Mounting Google Drive...')
drive.mount('/content/drive', force_remount=False)
DRIVE_ROOT = Path('/content/drive/MyDrive/MayzCats-Automation')
PROJECT_URL = 'https://github.com/m4mayz/mayzcats-automation.git'
RUNTIME_PROJECT = Path('/content/mayzcats-project')
RUNTIME_ROOT = Path('/content/mayzcats')
MPT_DIR = RUNTIME_ROOT / 'MoneyPrinterTurbo'

if RUNTIME_PROJECT.exists():
    shutil.rmtree(RUNTIME_PROJECT)
subprocess.run(['git', 'clone', '--depth', '1', PROJECT_URL, str(RUNTIME_PROJECT)], check=True)
print(f'[OK] Wrapper staged at {RUNTIME_PROJECT}')

In [ ]:
print('[MayzCats] Preparing FFmpeg, Montserrat, and Python dependencies...')
sys.path.insert(0, str(RUNTIME_PROJECT / 'src'))
from mayzcats.colab_bootstrap import ensure_system_dependencies  # noqa: E402

ensure_system_dependencies()
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(RUNTIME_PROJECT)], check=True)

import yaml  # noqa: E402

from mayzcats.storage import DriveLayout  # noqa: E402

layout = DriveLayout.bootstrap(DRIVE_ROOT, RUNTIME_PROJECT / 'config')
pipeline_config = yaml.safe_load(layout.pipeline_config.read_text(encoding='utf-8')) or {}
mpt = pipeline_config.get('video', {})
MPT_REPO = mpt.get('mpt_repo', 'https://github.com/harry0703/MoneyPrinterTurbo.git')
MPT_REF = mpt.get('mpt_ref', 'main')
RUNTIME_ROOT.mkdir(parents=True, exist_ok=True)
if (MPT_DIR / '.git').is_dir():
    subprocess.run(['git', '-C', str(MPT_DIR), 'fetch', '--depth', '1', 'origin', MPT_REF], check=True)
else:
    if MPT_DIR.exists():
        raise RuntimeError(f'{MPT_DIR} exists but is not a Git checkout')
    subprocess.run(['git', 'clone', '--depth', '1', MPT_REPO, str(MPT_DIR)], check=True)
    subprocess.run(['git', '-C', str(MPT_DIR), 'fetch', '--depth', '1', 'origin', MPT_REF], check=True)
subprocess.run(['git', '-C', str(MPT_DIR), 'checkout', '--detach', 'FETCH_HEAD'], check=True)
subprocess.run(['uv', 'python', 'install', '3.11'], check=True)
# MPT uses its untouched lockfile; equivalent command: uv sync --frozen --python 3.11
subprocess.run(['uv', 'sync', '--frozen', '--python', '3.11'], cwd=MPT_DIR, check=True)
print(f'[OK] MoneyPrinterTurbo ready at {MPT_DIR} ({MPT_REF})')

In [ ]:
from mayzcats.youtube_upload import load_credentials  # noqa: E402

print('[MayzCats] Connecting YouTube before paid pipeline stages...', flush=True)
load_credentials(layout.client_secret, layout.token_file)
print(f'[OK] YouTube OAuth token ready at {layout.token_file}', flush=True)

In [ ]:
def run_streamed(command):
    process = subprocess.Popen(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env={**os.environ, 'PYTHONUNBUFFERED': '1'},
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end='', flush=True)
    process.stdout.close()
    return process.wait()

print('[MayzCats] Running preflight before API usage...', flush=True)
preflight_code = run_streamed([
    sys.executable, '-u', '-m', 'mayzcats.preflight',
    '--drive-root', str(DRIVE_ROOT),
    '--mpt-root', str(MPT_DIR),
])
if preflight_code != 0:
    raise RuntimeError(f'MayzCats preflight failed with exit {preflight_code}; see details above.')

privacy = str(pipeline_config.get('youtube', {}).get('privacy', 'private')).lower()
pipeline_command = [
    sys.executable, '-u', '-m', 'mayzcats.pipeline',
    '--drive-root', str(DRIVE_ROOT),
    '--mpt-root', str(MPT_DIR),
    '--work-root', str(RUNTIME_ROOT / 'runs'),
]
if RUN_MODE == 'resume_latest':
    pipeline_command.append('--resume-latest')
elif RUN_MODE == 'resume':
    if not RESUME_RUN_ID.strip():
        raise ValueError('RESUME_RUN_ID is required when RUN_MODE is resume')
    pipeline_command.extend(['--resume', RESUME_RUN_ID.strip()])
elif RUN_MODE != 'new':
    raise ValueError('RUN_MODE must be new, resume_latest, or resume')
if RERENDER_ON_RESUME:
    if RUN_MODE not in {'resume', 'resume_latest'}:
        raise ValueError('RERENDER_ON_RESUME requires RUN_MODE resume or resume_latest')
    pipeline_command.append('--rerender')
print(f'[MayzCats] Starting mode={RUN_MODE}, YouTube privacy={privacy}...', flush=True)
pipeline_code = run_streamed(pipeline_command)
if pipeline_code != 0:
    raise RuntimeError(
        f'MayzCats stopped with exit {pipeline_code}. '
        f'The sanitized cause and run ID are printed above and saved under {DRIVE_ROOT / "logs"}.'
    )